# Подготовка датасета для Kaggle

Этот notebook создаёт компактный датасет для обучения ДОЭ: JPEG-изображения BSDS500, три воспроизводимых JSONL-манифеста и файл с метаданными. Синтетические изображения не материализуются: они создаются из seed во время обучения.

Запускайте его как Kaggle Notebook (не Docker). Если BSDS500 не прикреплён как Kaggle Input, включите Internet: будет скачан официальный архив, но в выходной набор попадёт только папка `images/`.

In [ ]:
from __future__ import annotations

import json
import math
import shutil
import tarfile
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset

# --- Configuration ---------------------------------------------------------
IMAGE_SIZE = 128
SPLIT_COUNTS = {'train': 24_000, 'val': 2_000, 'test_internal': 2_000}
BSDS_SPLITS = {'train': 'train', 'val': 'val', 'test_internal': 'test'}
SPLIT_SEEDS = {'train': 42, 'val': 31415, 'test_internal': 27182}
NATURAL_FRACTION = 0.40
OUTPUT_DIR = Path('/kaggle/working/ai_phonetics_data_v1')
ARCHIVE_PATH = Path('/kaggle/working/BSR_bsds500.tgz')
OFFICIAL_BSDS_URL = 'https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz'

# Set this to an existing Kaggle Input when the images are already attached.
# It must contain images/train, images/val and images/test.
ATTACHED_BSDS_DATA: Path | None = None
DOWNLOAD_IF_MISSING = True

assert IMAGE_SIZE == 128
assert 0.0 <= NATURAL_FRACTION <= 1.0


## 1. Извлечение только исходных JPEG

Для готового датасета не нужны человеческие сегментации BSDS500. Этот код извлекает из официального архива только `BSDS500/data/images/`.

In [ ]:
def _download(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = Request(url, headers={'User-Agent': 'ai-phonetics-kaggle-builder'})
    with urlopen(request, timeout=120) as response, destination.open('wb') as stream:
        shutil.copyfileobj(response, stream, length=1024 * 1024)


def _extract_images_only(archive: Path, destination_data: Path) -> None:
    prefix = 'BSR/BSDS500/data/images/'
    with tarfile.open(archive, 'r:gz') as tar:
        members = [member for member in tar.getmembers() if member.isfile() and member.name.startswith(prefix)]
        if not members:
            raise RuntimeError('BSDS500 archive does not contain expected image files')
        for member in members:
            relative = Path(member.name).relative_to('BSR/BSDS500/data')
            target = destination_data / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            source = tar.extractfile(member)
            if source is None:
                raise RuntimeError(f'Cannot read {member.name} from archive')
            with source, target.open('wb') as output:
                shutil.copyfileobj(source, output)


def prepare_bsds_images() -> Path:
    destination_data = OUTPUT_DIR / 'BSDS500' / 'data'
    destination_images = destination_data / 'images'
    if all((destination_images / split).is_dir() for split in ('train', 'val', 'test')):
        return destination_data
    if ATTACHED_BSDS_DATA is not None:
        source_images = Path(ATTACHED_BSDS_DATA) / 'images'
        if not all((source_images / split).is_dir() for split in ('train', 'val', 'test')):
            raise FileNotFoundError(f'Expected images/{{train,val,test}} below {ATTACHED_BSDS_DATA}')
        destination_data.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_images, destination_images, dirs_exist_ok=True)
        return destination_data
    if not DOWNLOAD_IF_MISSING:
        raise FileNotFoundError('Attach BSDS500 as a Kaggle Input or set DOWNLOAD_IF_MISSING=True')
    if not ARCHIVE_PATH.exists():
        print('Downloading official BSDS500 archive once...')
        _download(OFFICIAL_BSDS_URL, ARCHIVE_PATH)
    _extract_images_only(ARCHIVE_PATH, destination_data)
    ARCHIVE_PATH.unlink(missing_ok=True)
    return destination_data


BSDS_DATA = prepare_bsds_images()
for split in ('train', 'val', 'test'):
    print(f'{split}: {len(list((BSDS_DATA / 'images' / split).glob('*.jpg')))} JPEG files')


## 2. Загрузчик четырёх целей

В этой ячейке находится self-contained версия загрузчика. Она читает манифест, создаёт синтетику по seed, читает BSDS-кропы и нормирует **каждый** канал на собственный максимум.

In [ ]:
RESAMPLE = Image.Resampling.LANCZOS


def normalize_per_image(image: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    maximum = image.amax() if image.ndim == 2 else image.amax(dim=(-2, -1), keepdim=True)
    if torch.any(maximum <= eps):
        raise ValueError('All-black target image cannot be normalized')
    return image / maximum


def pil_to_tensor(image: Image.Image) -> torch.Tensor:
    image = image.convert('L')
    values = np.asarray(image, dtype=np.float32) / 255.0
    return normalize_per_image(torch.from_numpy(values))


def load_bsds_item(spec: dict, bsds_root: Path, image_size: int = IMAGE_SIZE) -> torch.Tensor:
    path = (bsds_root / spec['path']).resolve()
    root = bsds_root.resolve()
    if root not in path.parents or not path.is_file():
        raise FileNotFoundError(f"Invalid BSDS image path: {spec['path']}")
    with Image.open(path) as source:
        image = source.convert('L')
        x, y = int(spec['crop']['x']), int(spec['crop']['y'])
        image = image.crop((x, y, x + image_size, y + image_size))
    if image.size != (image_size, image_size):
        raise ValueError(f"Invalid crop for {spec['path']}")
    return pil_to_tensor(image)


def draw_glyph(canvas: Image.Image, rng: np.random.Generator, value: int) -> None:
    char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
    font_size = int(rng.integers(22, 66))
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', size=font_size)
    except OSError:
        font = ImageFont.load_default()
    glyph = Image.new('L', (font_size * 2, font_size * 2), 0)
    glyph_draw = ImageDraw.Draw(glyph)
    bbox = glyph_draw.textbbox((0, 0), char, font=font)
    glyph_draw.text((-bbox[0], -bbox[1]), char, fill=value, font=font)
    bbox = glyph.getbbox()
    if bbox is None:
        return
    glyph = glyph.crop(bbox).rotate(float(rng.uniform(-25, 25)), resample=Image.Resampling.BILINEAR, expand=True)
    x = int(rng.integers(0, max(1, canvas.width - glyph.width + 1)))
    y = int(rng.integers(0, max(1, canvas.height - glyph.height + 1)))
    canvas.paste(glyph, (x, y), glyph)


def make_synthetic(seed: int, image_size: int = IMAGE_SIZE) -> torch.Tensor:
    rng = np.random.default_rng(seed)
    canvas = Image.new('L', (image_size, image_size), 0)
    draw = ImageDraw.Draw(canvas)
    for _ in range(int(rng.integers(1, 5))):
        kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'point', 'glyph']))
        value, width = int(rng.integers(80, 256)), int(rng.integers(1, 7))
        x0, x1 = sorted(rng.integers(4, image_size - 4, size=2).tolist())
        y0, y1 = sorted(rng.integers(4, image_size - 4, size=2).tolist())
        x1, y1 = max(x1, x0 + 8), max(y1, y0 + 8)
        x1, y1 = min(x1, image_size - 1), min(y1, image_size - 1)
        if kind == 'ellipse':
            draw.ellipse((x0, y0, x1, y1), fill=value if rng.random() < 0.5 else None, outline=value, width=width)
        elif kind == 'rectangle':
            draw.rectangle((x0, y0, x1, y1), fill=value if rng.random() < 0.5 else None, outline=value, width=width)
        elif kind == 'line':
            draw.line((x0, y0, x1, y1), fill=value, width=width)
        elif kind == 'ring':
            draw.ellipse((x0, y0, x1, y1), outline=value, width=width)
        elif kind == 'point':
            radius = int(rng.integers(2, 10))
            cx, cy = int(rng.integers(radius, image_size - radius)), int(rng.integers(radius, image_size - radius))
            draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=value)
        else:
            draw_glyph(canvas, rng, value)
    return pil_to_tensor(canvas)


class QuadDataset(Dataset):
    def __init__(self, manifest_path: Path, bsds_root: Path):
        with Path(manifest_path).open(encoding='utf-8') as stream:
            self.records = [json.loads(line) for line in stream if line.strip()]
        self.bsds_root = Path(bsds_root)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        items = self.records[index]['items']
        assert len(items) == 4
        images = [make_synthetic(int(item['seed'])) if item['source'] == 'synthetic'
                  else load_bsds_item(item, self.bsds_root) for item in items]
        return torch.stack(images).to(torch.float32)


## 3. Генерация детерминированных манифестов

Манифест хранит только спецификацию четырёх изображений. Поэтому 28 000 четвёрок занимают считанные мегабайты, а не гигабайты PNG.

In [ ]:
def image_catalog(bsds_root: Path, split: str) -> list[tuple[str, int, int]]:
    result = []
    for path in sorted((bsds_root / 'images' / split).glob('*.jpg')):
        with Image.open(path) as image:
            width, height = image.size
        if min(width, height) >= IMAGE_SIZE:
            result.append((path.relative_to(bsds_root).as_posix(), width, height))
    if not result:
        raise RuntimeError(f'No valid JPEG files found for BSDS split {split!r}')
    return result


def sample_item(rng: np.random.Generator, catalog: list[tuple[str, int, int]]) -> dict:
    if rng.random() >= NATURAL_FRACTION:
        return {'source': 'synthetic', 'seed': int(rng.integers(1, 2**63 - 1))}
    path, width, height = catalog[int(rng.integers(0, len(catalog)))]
    return {
        'source': 'bsds',
        'path': path,
        'crop': {
            'x': int(rng.integers(0, width - IMAGE_SIZE + 1)),
            'y': int(rng.integers(0, height - IMAGE_SIZE + 1)),
        },
    }


def make_manifest(name: str, count: int, bsds_split: str, seed: int) -> Path:
    rng = np.random.default_rng(seed)
    catalog = image_catalog(BSDS_DATA, bsds_split)
    output = OUTPUT_DIR / 'manifests' / f'{name}.jsonl'
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open('w', encoding='utf-8') as stream:
        for index in range(count):
            items, signatures = [], set()
            while len(items) < 4:
                item = sample_item(rng, catalog)
                signature = json.dumps(item, sort_keys=True)
                if signature not in signatures:
                    signatures.add(signature)
                    items.append(item)
            stream.write(json.dumps({'id': f'{name}-{index:05d}', 'items': items}) + '\n')
    return output


manifests = {
    name: make_manifest(name, count, BSDS_SPLITS[name], SPLIT_SEEDS[name])
    for name, count in SPLIT_COUNTS.items()
}
for name, path in manifests.items():
    print(f'{name}: {sum(1 for _ in path.open())} quads -> {path}')


## 4. Проверка loader и упаковка

Проверяется только несколько строк каждого манифеста: форма `(4, 128, 128)`, `float32`, диапазон `[0, 1]` и максимум 1 в каждом канале. Затем готовый набор архивируется в `/kaggle/working/ai_phonetics_data_v1.zip`.

In [ ]:
for name, manifest in manifests.items():
    dataset = QuadDataset(manifest, BSDS_DATA)
    for index in (0, len(dataset) // 2, len(dataset) - 1):
        quad = dataset[index]
        assert quad.shape == (4, 128, 128)
        assert quad.dtype == torch.float32
        assert torch.all((quad >= 0) & (quad <= 1))
        assert torch.allclose(quad.amax(dim=(-2, -1)), torch.ones(4))
    print(f'{name}: loader check passed')

metadata = {
    'schema_version': 1,
    'image_size': IMAGE_SIZE,
    'split_counts': SPLIT_COUNTS,
    'natural_fraction': NATURAL_FRACTION,
    'synthetic_fraction': 1.0 - NATURAL_FRACTION,
    'split_seeds': SPLIT_SEEDS,
    'bsds_source_url': OFFICIAL_BSDS_URL,
    'bsds_image_counts': {split: len(list((BSDS_DATA / 'images' / split).glob('*.jpg'))) for split in ('train', 'val', 'test')},
}
(OUTPUT_DIR / 'dataset_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/ai_phonetics_data_v1', 'zip', root_dir=OUTPUT_DIR)
print(f'Created {archive}: {Path(archive).stat().st_size / 1024**2:.1f} MiB')
print('Use Save Version in Kaggle to retain this file, then create a Kaggle Dataset from the notebook output.')
